In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import  Distance, VectorParams, PointStruct
from dotenv import load_dotenv
import os, openai

import openai
import pandas as pd

load_dotenv()  # reads .env in project root
openai.api_key = os.getenv("OPENAI_API_KEY")


In [4]:
qdrant_client = QdrantClient(url="http://localhost:6333")

qdrant_client.create_collection(
    collection_name="Amazon-items-collections-00",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

UnexpectedResponse: Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `Amazon-items-collections-00` already exists!"},"time":0.003195542}'

In [5]:
df_items = pd.read_json("../notebook/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [6]:
df_items.head(2)

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Industrial & Scientific,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",4.4,119,[【Fast Charging Cord】These USB C cables provid...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Type-C Charger Cable ', 'url': 'ht...",RAVODOI,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'RAVODOI', 'Connector Type': 'USB Ty...",B09R4Y2HKY,NaN,NaN,NaN
1,All Electronics,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.5,352,[🔹(Light & compact) Easy to carry and light we...,[],4.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'USB Male & Female Adapter', 'url':...",SNESH,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '3.54 x 2.4 x 0.35 inch...,B09JV5FM2S,NaN,NaN,NaN


In [7]:
def preprocess_data(row):
    return f"{row['title']} {' '.join(row['description'])}"

In [8]:
df_items["preprocessed_data"] = df_items.apply(preprocess_data, axis=1)

In [9]:
df_items.head(2)

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,preprocessed_data
0,Industrial & Scientific,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",4.4,119,[【Fast Charging Cord】These USB C cables provid...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Type-C Charger Cable ', 'url': 'ht...",RAVODOI,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'RAVODOI', 'Connector Type': 'USB Ty...",B09R4Y2HKY,NaN,NaN,NaN,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T..."
1,All Electronics,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.5,352,[🔹(Light & compact) Easy to carry and light we...,[],4.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'USB Male & Female Adapter', 'url':...",SNESH,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '3.54 x 2.4 x 0.35 inch...,B09JV5FM2S,NaN,NaN,NaN,"SNESH-2 Pack USB-C Female to USB Male Adapter,..."


In [10]:
df_sample = df_items.sample(n=50, random_state=42)

In [11]:
def get_embedding(text, model="text-embedding-ada-002"):
    response = openai.embeddings.create(
        input=[text],
        model=model,
    )   
    return response.data[0].embedding
    

In [12]:
get_embedding("hi, my name is rahul")

[-0.013956214301288128,
 -0.004416482988744974,
 -0.011967037804424763,
 -0.01803690381348133,
 -0.021516362205147743,
 0.01772989332675934,
 -0.028091518208384514,
 -0.01959754340350628,
 -0.021145392209291458,
 -0.006172202993184328,
 0.022258305922150612,
 -0.019367285072803497,
 0.0032971715554594994,
 0.00244809384457767,
 -0.0016837641596794128,
 -0.027426326647400856,
 0.019815009087324142,
 -0.01796015165746212,
 0.02511095255613327,
 0.004259779583662748,
 0.0007227553869597614,
 -0.005139238201081753,
 0.022296683862805367,
 -0.0035338259767740965,
 -0.020595328882336617,
 -0.012210088782012463,
 0.02804034948348999,
 -0.012638624757528305,
 0.009044036269187927,
 -0.03348979726433754,
 -0.006463223602622747,
 -0.004908979870378971,
 -0.007815991528332233,
 -0.01569594442844391,
 0.006009103264659643,
 -0.00947257224470377,
 0.008385241031646729,
 -0.010176139883697033,
 0.012625833041965961,
 0.016565809026360512,
 0.03461550548672676,
 -0.00882656965404749,
 0.0062169753946

In [13]:
data_to_embed = df_sample["preprocessed_data"].to_list()
pointstruct_list = []
for i, data in enumerate(data_to_embed):
    pointstruct_list.append(
        PointStruct(id=i, vector=get_embedding(data), payload={"text": data})
    )   

In [58]:
pointstruct_list

[PointStruct(id=0, vector=[-0.024218503385782242, -0.0018264788668602705, 0.015311476774513721, -0.027743641287088394, -0.005758622195571661, 0.005570255685597658, -0.022698119282722473, -0.02723236195743084, 0.007305915467441082, -0.018984615802764893, 0.0337444469332695, 0.024124320596456528, -0.02335740067064762, 0.010938690975308418, -0.009317396208643913, -0.003972507547587156, 0.002430259482935071, 0.00897430069744587, 0.020653001964092255, 0.007144458591938019, -0.01967080682516098, 0.023976318538188934, 0.0023713952396064997, -0.00722518702968955, -0.019912991672754288, -0.01107996515929699, 0.019899537786841393, -0.004520787391811609, 0.004238238092511892, -0.014302372001111507, 0.01626676134765148, 0.015217293053865433, -0.004140691366046667, -0.01449073851108551, -0.03105350397527218, 0.014948198571801186, -0.009539399296045303, -0.014786741696298122, 0.017248956486582756, 0.013750728219747543, 0.015795845538377762, 0.010945417918264866, -0.00061218993505463, 0.0171951372176

In [14]:
qdrant_client.upsert(collection_name="Amazon-items-collections-00", wait=True, points=pointstruct_list)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [15]:
def retrieve_data(query):
    query_embedding = get_embedding(query)
    results = qdrant_client.search(
        collection_name="Amazon-items-collections-00",
        query_vector=query_embedding,
        limit=5,
    )
    return results

In [17]:
retrieve_data("WiFi Range Extender").points

/var/folders/q4/llynm5vn2t38psmszbvhc82r0000gn/T/ipykernel_24709/665691538.py:3: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = qdrant_client.search(


AttributeError: 'list' object has no attribute 'points'